# 03 — Projection Uncertainty & Robust Lineup Selection

**What you'll learn:**
- Why a single "optimal" lineup can be fragile — projections are wrong!
- How Monte Carlo simulation reveals the distribution of possible outcomes
- How to identify players who appear in optimal lineups most often ("core" players)
- How to compare ceiling vs floor across different lineup strategies
- How to build a **robust** lineup that performs well across many scenarios

**Prerequisites:** FFPy installed, `numpy`, `matplotlib`, `seaborn`. No database needed.

## Setup

This notebook uses FFPy's optimizer and scoring modules. No database required —
we construct sample players in-memory.

```python
# Run this cell to install analysis deps (first time only)
# !uv sync --group analysis
```

## Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

_repo = Path.cwd()
while not (_repo / "pyproject.toml").exists() and _repo.parent != _repo:
    _repo = _repo.parent
sys.path.insert(0, str(_repo / "src"))

from ffpy.optimizer import (
    LineupOptimizer, LineupResult,
    Player, PlayerStatus, RosterConstraints,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## Data: Player Pool

In [ ]:
def make_pool():
    return [
        # ── QB ──
        Player("Josh Allen",       "QB", "BUF", 25.8, consistency=4.2),
        Player("Patrick Mahomes",  "QB", "KC",  24.5, consistency=3.8),
        Player("Lamar Jackson",    "QB", "BAL", 24.2, consistency=5.1),
        Player("Jalen Hurts",      "QB", "PHI", 23.1, consistency=4.5),
        Player("Joe Burrow",       "QB", "CIN", 22.0, consistency=3.5),
        Player("Dak Prescott",     "QB", "DAL", 21.5, consistency=4.0),
        # ── RB ──
        Player("Christian McCaffrey","RB", "SF", 22.4, consistency=6.0),
        Player("Saquon Barkley",   "RB", "PHI", 20.1, consistency=5.5),
        Player("Bijan Robinson",   "RB", "ATL", 19.2, consistency=5.0),
        Player("Austin Ekeler",    "RB", "WAS", 18.7, consistency=4.8),
        Player("Derrick Henry",    "RB", "BAL", 17.5, consistency=6.5),
        Player("James Cook",       "RB", "BUF", 15.2, consistency=4.0),
        # ── WR ──
        Player("Tyreek Hill",      "WR", "MIA", 19.8, consistency=5.2),
        Player("CeeDee Lamb",      "WR", "DAL", 18.5, consistency=4.5),
        Player("Justin Jefferson", "WR", "MIN", 18.2, consistency=4.0),
        Player("Amon-Ra St. Brown","WR", "DET", 16.9, consistency=3.5),
        Player("Stefon Diggs",     "WR", "HOU", 14.3, consistency=4.2),
        Player("Davante Adams",    "WR", "LV",  14.8, consistency=5.0),
        # ── TE ──
        Player("Travis Kelce",     "TE", "KC",  16.5, consistency=4.5),
        Player("Mark Andrews",     "TE", "BAL", 14.2, consistency=4.0),
        Player("George Kittle",    "TE", "SF",  13.1, consistency=5.5),
        Player("Sam LaPorta",      "TE", "DET", 12.5, consistency=3.5),
        # ── K ──
        Player("Justin Tucker",    "K",  "BAL",  9.5, consistency=2.0),
        Player("Harrison Butker",  "K",  "KC",   9.2, consistency=2.5),
        # ── DST ──
        Player("49ers DST",        "DST","SF",  10.2, consistency=3.0),
        Player("Ravens DST",       "DST","BAL",  9.5, consistency=3.5),
        Player("Cowboys DST",      "DST","DAL",  8.8, consistency=4.0),
    ]

base_pool = make_pool()
print(f"{len(base_pool)} players")


### 1. Understanding Projection Variance

The `Player.consistency` attribute stores the standard deviation of recent
scores. Players with high consistency (std dev) are boom-or-bust — they might
score 30 or 8. Low-consistency players are reliable.

Let's visualise the distribution of possible outcomes:

In [ ]:
rng = np.random.default_rng(42)
outcomes = 5000

# Compare a high-consistency vs low-consistency player
reliable = {"name": "Amon-Ra St. Brown", "proj": 16.9, "std": 3.5}
volatile = {"name": "Derrick Henry",     "proj": 17.5, "std": 6.5}

reliable_scores = rng.normal(reliable["proj"], reliable["std"], outcomes)
volatile_scores = rng.normal(volatile["proj"], volatile["std"], outcomes)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(reliable_scores, bins=60, alpha=0.6, label=f"{reliable['name']} (σ={reliable['std']})", density=True)
ax.hist(volatile_scores, bins=60, alpha=0.6, label=f"{volatile['name']} (σ={volatile['std']})", density=True)
ax.axvline(reliable["proj"], color="C0", ls="--", alpha=0.5)
ax.axvline(volatile["proj"], color="C1", ls="--", alpha=0.5)
ax.set_xlabel("Fantasy Points")
ax.set_ylabel("Density")
ax.set_title("Outcome Distribution: Reliable vs Volatile Player")
ax.legend()
plt.show()


**Insight:** The volatile player has higher upside but also deeper downside.
A single "optimal" lineup ignores this — it treats `17.5` as a certainty.


### 2. Monte Carlo Simulation Framework

We'll simulate the "true" outcome for each player by sampling from a normal
distribution centered on their projection with their consistency as σ.
Then we re-optimize the lineup with each simulated outcome set.

**The question:** Which players appear in the optimal lineup most often?

In [ ]:
def monte_carlo_optimize(pool, n_sims=1000, seed=42):
    """
    Run n_sims Monte Carlo simulations:
    - For each sim, draw each player's "true" score from N(proj, consistency^2)
    - Re-optimize the lineup
    - Track which players are selected
    Returns: selection_counts, avg_lineup_score, lineup_scores
    """
    rng = np.random.default_rng(seed)
    constraints = RosterConstraints.standard()

    # Track how often each player is selected
    selection_counter = Counter()
    lineup_scores = []

    for sim in range(n_sims):
        # Create simulated player pool with perturbed projections
        sim_pool = []
        for p in pool:
            # If consistency is 0 or None, use 15% of projection as std
            std = p.consistency if (p.consistency and p.consistency > 0) else p.projected_points * 0.15
            simulated_pts = round(float(rng.normal(p.projected_points, std)), 1)
            sim_pool.append(Player(
                name=p.name,
                position=p.position,
                team=p.team,
                projected_points=max(0.1, simulated_pts),  # floor at 0.1
                consistency=p.consistency,
            ))

        try:
            opt = LineupOptimizer(constraints)
            res = opt.optimize(sim_pool)
            for starter in res.starters:
                selection_counter[starter.name] += 1
            lineup_scores.append(res.total_points)
        except ValueError:
            lineup_scores.append(0.0)  # Infeasible — shouldn't happen

    return selection_counter, lineup_scores

n_sims = 1000
counter, scores = monte_carlo_optimize(base_pool, n_sims=n_sims)
print(f"Ran {n_sims} simulations")

# Show selection frequency
selection_df = pd.DataFrame([
    {"player": name, "selection_pct": round(count / n_sims * 100, 1), "times_selected": count}
    for name, count in counter.most_common()
])
print(f"\n=== Player Selection Frequency (top 15) ===")
print(selection_df.head(15).to_string(index=False))


### 3. Visualize Selection Frequencies

In [ ]:
# Add position info
pos_map = {p.name: p.position for p in base_pool}
selection_df["position"] = selection_df["player"].map(pos_map)

fig, ax = plt.subplots(figsize=(12, 6))
colors = {"QB": "#1f77b4", "RB": "#ff7f0e", "WR": "#2ca02c", "TE": "#d62728",
          "K": "#9467bd", "DST": "#8c564b"}

top20 = selection_df.head(20).sort_values("selection_pct")
ax.barh(range(len(top20)), top20["selection_pct"],
        color=[colors[p] for p in top20["position"]])
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["player"])
ax.set_xlabel("Selection Frequency (%)")
ax.set_title(f"Player Selection Frequency across {n_sims} Monte Carlo Simulations")
ax.legend(handles=[
    plt.Rectangle((0, 0), 1, 1, color=c, label=p)
    for p, c in colors.items()
], title="Position")
plt.tight_layout()
plt.show()

print("Players with >90% selection rate (core starters):")
core = selection_df[selection_df["selection_pct"] > 90]
if not core.empty:
    for _, row in core.iterrows():
        print(f"  • {row['player']:25} {row['position']:3}  ({row['selection_pct']:.0f}%)")
else:
    print("  (none — high uncertainty across the board)")


### 4. Lineup Score Distribution

Now let's look at the distribution of total lineup scores across all simulations.
This tells us the expected range of outcomes:

In [ ]:
scores_arr = np.array(scores)
scores_arr = scores_arr[scores_arr > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(scores_arr, bins=40, edgecolor="white", alpha=0.7)
axes[0].axvline(np.mean(scores_arr), color="r", ls="--", label=f"Mean: {np.mean(scores_arr):.1f}")
axes[0].axvline(np.percentile(scores_arr, 5), color="orange", ls=":", label=f"5th: {np.percentile(scores_arr, 5):.1f}")
axes[0].axvline(np.percentile(scores_arr, 95), color="orange", ls=":", label=f"95th: {np.percentile(scores_arr, 95):.1f}")
axes[0].set_xlabel("Total Projected Points")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Optimal Lineup Scores")
axes[0].legend()

# Cumulative distribution
sorted_scores = np.sort(scores_arr)
cdf = np.arange(1, len(sorted_scores) + 1) / len(sorted_scores)
axes[1].plot(sorted_scores, cdf)
axes[1].axhline(0.5, color="gray", ls="--", alpha=0.5)
axes[1].axvline(np.median(scores_arr), color="r", ls="--", label=f"Median: {np.median(scores_arr):.1f}")
axes[1].set_xlabel("Total Projected Points")
axes[1].set_ylabel("Cumulative Probability")
axes[1].set_title("CDF of Optimal Lineup Scores")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Score stats across {len(scores_arr)} simulations:")
print(f"  Mean:   {np.mean(scores_arr):.1f}")
print(f"  Median: {np.median(scores_arr):.1f}")
print(f"  Std:    {np.std(scores_arr):.1f}")
print(f"  5th-95th range: {np.percentile(scores_arr, 5):.1f} – {np.percentile(scores_arr, 95):.1f}")
print(f"  Min:    {scores_arr.min():.1f}")
print(f"  Max:    {scores_arr.max():.1f}")


### 5. Ceiling vs Floor: Comparing Lineup Strategies

Let's compare two strategies:
1. **Point-maximizing** — the standard optimizer (maximise sum of projections)
2. **Floor-maximizing** — prefer low-consistency (reliable) players, trading
   some upside for stability

We'll implement floor-maximising by lowering the projection of high-consistency
players before optimisation:

In [ ]:
def floor_optimize(pool):
    """Penalize volatile players: adj_proj = proj - 0.5 * consistency"""
    adj_pool = []
    for p in pool:
        penalty = 0.5 * (p.consistency if p.consistency else p.projected_points * 0.15)
        adj_pool.append(Player(
            name=p.name, position=p.position, team=p.team,
            projected_points=max(0.1, p.projected_points - penalty),
            consistency=p.consistency,
        ))
    opt = LineupOptimizer(RosterConstraints.standard())
    return opt.optimize(adj_pool)

# Run both strategies
pt_opt = LineupOptimizer(RosterConstraints.standard())
max_res = pt_opt.optimize(base_pool)

floor_res = floor_optimize(base_pool)

print("=== Point-Maximising (Standard) ===")
print(pt_opt.analyze_lineup(max_res))
print()
print("=== Floor-Maximising (Consistency-Adjusted) ===")
print(pt_opt.analyze_lineup(floor_res))

# Simulate both lineups against the Monte Carlo outcomes
def simulate_lineup(lineup_players, pool, n_sims=5000):
    rng = np.random.default_rng(99)
    scores = []
    for _ in range(n_sims):
        total = 0
        for p in lineup_players:
            std = p.consistency if p.consistency and p.consistency > 0 else p.projected_points * 0.15
            total += max(0.1, rng.normal(p.projected_points, std))
        scores.append(total)
    return np.array(scores)

max_scores = simulate_lineup(max_res.starters, base_pool)
floor_scores = simulate_lineup(floor_res.starters, base_pool)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_scores, bins=50, alpha=0.5, label="Point-Maximising", density=True)
ax.hist(floor_scores, bins=50, alpha=0.5, label="Floor-Maximising", density=True)
ax.axvline(np.mean(max_scores), color="C0", ls="--")
ax.axvline(np.mean(floor_scores), color="C1", ls="--")
ax.set_xlabel("Total Fantasy Points")
ax.set_ylabel("Density")
ax.set_title("Simulated Outcome Distribution: Two Strategies")
ax.legend()
plt.show()

print()
print(f"{'Metric':<20} {'Point-Max':>10} {'Floor-Max':>10}")
print("-" * 42)
print(f"{'Mean':<20} {np.mean(max_scores):>10.1f} {np.mean(floor_scores):>10.1f}")
print(f"{'Std Dev':<20} {np.std(max_scores):>10.1f} {np.std(floor_scores):>10.1f}")
print(f"{'5th %ile':<20} {np.percentile(max_scores, 5):>10.1f} {np.percentile(floor_scores, 5):>10.1f}")
print(f"{'95th %ile':<20} {np.percentile(max_scores, 95):>10.1f} {np.percentile(floor_scores, 95):>10.1f}")
print(f"{'Floor risk (5th)':<20} {np.percentile(max_scores, 5):>10.1f} {np.percentile(floor_scores, 5):>10.1f}")


**What to look for:**
- The point-max lineup has a higher *mean* but also a heavier left tail (worse floor)
- The floor-max lineup has a narrower distribution (less variance)
- In a winner-take-all contest, ceiling matters more. In a weekly head-to-head
  league, floor matters more.


### 6. Robust Lineup Construction

A "robust" lineup is one that performs well *consistently* across many possible
outcome scenarios. The players selected most often in the Monte Carlo simulation
are the most robust — they're optimal under many different projection realizations.

Let's build a robust lineup from the top-9 most-frequently-selected players:

In [ ]:
# Get the most-frequently selected players, respecting position constraints
def build_robust_lineup(selection_df, pool):
    """Build lineup respecting position constraints from selection frequency."""
    pos_map = {p.name: p.position for p in pool}
    proj_map = {p.name: p.projected_points for p in pool}

    # Sort by selection frequency
    sorted_players = selection_df.sort_values("selection_pct", ascending=False)
    sorted_players["position"] = sorted_players["player"].map(pos_map)

    constraints = RosterConstraints.standard()
    pos_counts = {}
    starters = []

    for _, row in sorted_players.iterrows():
        pos = row["position"]
        max_at_pos = constraints.positions.get(pos, 0)
        current_at_pos = pos_counts.get(pos, 0)

        # Check if position slot available
        if current_at_pos < max_at_pos:
            starters.append(row["player"])
            pos_counts[pos] = current_at_pos + 1

        # Check FLEX
        elif pos in constraints.flex_positions and pos_counts.get("FLEX", 0) < constraints.num_flex:
            # Count how many are already in flex-eligible positions beyond base
            flex_positions = constraints.flex_positions
            base_total = sum(constraints.positions.get(p, 0) for p in flex_positions)
            flex_used = sum(pos_counts.get(p, 0) for p in flex_positions) - base_total
            if flex_used < constraints.num_flex:
                starters.append(row["player"])
                pos_counts[pos] = current_at_pos + 1

        if len(starters) >= constraints.total_starters:
            break

    # Return Player objects
    name_map = {p.name: p for p in pool}
    return [name_map[name] for name in starters]

robust_starters = build_robust_lineup(selection_df, base_pool)
robust_total = sum(p.projected_points for p in robust_starters)
robust_vs_pointmax = max_res.total_points - robust_total

print("=== Robust Lineup (most-selected players) ===")
for p in robust_starters:
    pct = counter.get(p.name, 0) / n_sims * 100
    print(f"  {p.name:25} {p.position:3} {p.team:4}  {p.projected_points:5.1f}  (selected {pct:.0f}%)")
print(f"  {'─'*55}")
print(f"  {'Total':>25}        {robust_total:5.1f}")
print()
print(f"Point-max total:       {max_res.total_points:.1f}")
print(f"Robust total:          {robust_total:.1f}")
print(f"Cost of robustness:    {robust_vs_pointmax:+.1f} pts")
print()
print("The robust lineup trades a few points of expected value for")
print("higher confidence that the lineup will perform well in practice.")


### 7. Your Turn: Sensitivity to Projections

Try changing a player's projection and re-running the Monte Carlo simulation.
How much does the optimal lineup change?

In [ ]:
def sensitivity_test(player_name, new_projection, pool, n_sims=500):
    """Test how changing one player's projection affects selection frequencies."""
    modified_pool = []
    for p in pool:
        if p.name == player_name:
            modified_pool.append(Player(
                p.name, p.position, p.team, new_projection, consistency=p.consistency,
            ))
        else:
            modified_pool.append(p)

    counter_mod, _ = monte_carlo_optimize(modified_pool, n_sims=n_sims, seed=42)
    return counter_mod

# Example: What if Christian McCaffrey is projected for 28 pts instead of 22.4?
cmc_boosted = sensitivity_test("Christian McCaffrey", 28.0, base_pool)
cmc_pct = cmc_boosted.get("Christian McCaffrey", 0) / 500 * 100

original_pct = counter.get("Christian McCaffrey", 0) / n_sims * 100
print(f"CMC at 22.4 pts: selected in {original_pct:.0f}% of lineups")
print(f"CMC at 28.0 pts: selected in {cmc_pct:.0f}% of lineups")
print()

# See who drops when CMC rises
players_dropped = {}
for name, count in counter.items():
    new_count = cmc_boosted.get(name, 0)
    players_dropped[name] = (count / n_sims * 100) - (new_count / 500 * 100)

dropped_sorted = sorted(players_dropped.items(), key=lambda x: x[1])
print("Players whose selection rate dropped most when CMC's projection rose:")
for name, delta in dropped_sorted[:5]:
    print(f"  {name:25} {delta:+.1f}%")


### Key Takeaways

1. **Projections are not certainties** — every projection has uncertainty.
   The `consistency` (std dev) captures this.
2. **Monte Carlo simulation reveals robustness** — players selected in >90% of
   simulated lineups are "core" starters.
3. **Point-maximising ≠ best strategy** — depending on your league format
   (head-to-head vs total points), you may prefer a higher floor or higher ceiling.
4. **The robust lineup option** trades expected points for reduced variance —
   useful in weekly head-to-head leagues where consistency beats boom-or-bust.
5. **Sensitivity analysis** shows how lineup construction depends on your
   projections — a small change in one player's projection can reshape the
   entire optimal lineup.


---
*Notebook generated by `scripts/generate_optimization_notebooks.py`.*
